# Homework 2: Agentic RAG

We build a RAG system over the course lesson pages, then make it agentic.

## Preparation: Fetch lesson pages from GitHub

Data loading and index building live in [ingest.py](ingest.py):
- `load_documents()` — downloads lesson pages from GitHub at commit `8c1834d`
- `build_index(documents)` — builds a full-document minsearch index
- `build_chunk_index(documents)` — splits pages into overlapping 2000-char chunks and indexes them

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

from ingest import load_documents, build_index, build_chunk_index
from rag import LessonRAG

load_dotenv()
client = OpenAI()

In [2]:
documents = load_documents()

## Q1. How many lesson pages?

Each document = one lesson page.

In [3]:
print(f"Number of lesson pages: {len(documents)}")

Number of lesson pages: 72


## Q2. Indexing and searching

`build_index` creates a minsearch index with `content` as a text field (full-text scored) and `filename` as a keyword field (exact-match filter).

In [4]:
index = build_index(documents)

In [5]:
query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(query, num_results=5)

for r in results:
    print(r["filename"])

01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/11-agents-intro.md
01-agentic-rag/lessons/16-other-frameworks.md


In [6]:
print(f"Q2 answer (top result): {results[0]['filename']}")

Q2 answer (top result): 01-agentic-rag/lessons/14-agentic-loop.md


## Q3. RAG

**RAG = Retrieve → Augment → Generate**

1. **Retrieve**: search the index for relevant docs
2. **Augment**: stuff those docs into a prompt as context
3. **Generate**: send the prompt to an LLM and get an answer

`LessonRAG` (in [rag.py](rag.py)) subclasses `RAGBase` and adapts it for our `filename`/`content` schema. Its `rag()` returns `(answer, usage)` so we can read `usage.input_tokens`.

In [7]:
rag = LessonRAG(index=index, llm_client=client, model="gpt-4o-mini")

answer, usage = rag.rag("How does the agentic loop keep calling the model until it stops?")

print(answer)
print()
print(f"Input tokens: {usage.input_tokens}")

The agentic loop keeps calling the model until it receives a response without any function calls. This is how it works:

1. The loop is set up with a `while True` structure that continues executing until a specified condition is met.
2. Each iteration of the loop involves sending a set of messages (which include the developer's instructions and the user's question) to the model, alongside the available tools.
3. After receiving a response, the loop checks if the response includes any function calls (e.g., to the search function).
4. If there are function calls present, the loop executes those calls, appends the results to the conversation, and sets a flag (`has_function_calls`) indicating that another call is necessary.
5. The iteration counter is incremented, and the loop continues to run until a response is generated that does not contain any function calls, at which point it breaks out of the loop.

The key mechanism is that the model decides when to stop based on whether it require

## Q4. Chunking

Long documents hurt retrieval: a match buried deep in a page still pulls the whole page into context. `build_chunk_index` in [ingest.py](ingest.py) slices each document with a sliding window (`size=2000`, `step=1000`) so consecutive chunks overlap by 1000 chars, then indexes them.

In [8]:
chunks, chunk_index = build_chunk_index(documents)

print(f"Number of chunks: {len(chunks)}")

Number of chunks: 295


## Q5. RAG with chunking

Now we index the chunks instead of full documents and run the same query.
Because each retrieved chunk is much shorter than a full page, the prompt we send to
the LLM is smaller — we measure that with `usage.input_tokens`.

In [9]:
rag_chunked = LessonRAG(index=chunk_index, llm_client=client, model="gpt-4o-mini")

answer_chunked, usage_chunked = rag_chunked.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

print(answer_chunked)
print()
print(f"Input tokens (chunked): {usage_chunked.input_tokens}")

The agentic loop continues to call the model in a `while True` structure, where it checks for function calls in each iteration. It retrieves responses from the model and appends them to the message history. If the model's response includes function calls, the loop executes those functions and updates the message history again. This process repeats until the model provides a response that does not include any function calls, at which point the loop breaks and it stops calling the model. The exit condition for the loop is simply that no function calls are present in the current response.

Input tokens (chunked): 2310


In [10]:
print(f"Full-doc input tokens : {usage.input_tokens}")
print(f"Chunked input tokens  : {usage_chunked.input_tokens}")
print(f"Ratio                 : {usage.input_tokens / usage_chunked.input_tokens:.1f}x fewer")

Full-doc input tokens : 7127
Chunked input tokens  : 2310
Ratio                 : 3.1x fewer


## Q6. Turning it into an agent

So far we ran exactly one search per question. An **agent** is different:
- We give the LLM a `search` *tool* (a callable function)
- The LLM decides *when* to call it and *what* to search for
- This loops until the model produces a final answer (no more tool calls)

We use `toyaikit` — the small agent library from the module.
The library reads the function's type hints and docstring to build the JSON schema
that gets sent to the LLM as the tool definition.

In [11]:
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools

search_call_count = 0

def search(query: str) -> list:
    """Search the course lesson pages for information relevant to the query."""
    global search_call_count
    search_call_count += 1
    return chunk_index.search(query, num_results=5)


AGENT_INSTRUCTIONS = (
    "You're a course teaching assistant. Answer the student's question using the "
    "search tool. Make multiple searches with different keywords before answering."
)

tools = Tools()
tools.add_tool(search)

llm_client = OpenAIClient(model="gpt-4o-mini", client=client)

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=AGENT_INSTRUCTIONS,
    llm_client=llm_client,
    chat_interface=None,
)

In [12]:
search_call_count = 0

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=None,
)

print(result.last_message)
print()
print(f"search() called {search_call_count} times")

The **agentic loop** and **Retrieval-Augmented Generation (RAG)** are distinct concepts in the realm of AI workflows, particularly in how they manage interactions and data retrieval.

### Agentic Loop
The agentic loop refers to a specific operational framework designed for building AI agents that autonomously manage tasks and conversations. It involves:
1. **Continuous Interaction**: The agent operates in a loop, continuously calling the LLM (Large Language Model) and executing functions until it reaches a final answer. This means that the agent can inquire multiple times or correct itself if it doesn't retrieve satisfactory results on the first try.
2. **Three Key Components**:
   - **Instructions**: Defines the role and behaviors expected from the agent.
   - **Tools**: Various functions that the agent can use to complete tasks.
   - **Memory**: Keeps track of past interactions to inform future queries and actions.
3. **Dynamic Decision-Making**: The agent decides how many times to s